In [14]:
import yfinance as yf
import pandas as pd
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

# ── Load data ──────────────────────────────────────────────────────────────────
df = yf.download("NQ=F", interval="5m", period="60d", auto_adjust=True)
df.columns = df.columns.get_level_values(0)
df = df.dropna()

opens  = df["Open"].values.astype(np.float32).ravel()
closes = df["Close"].values.astype(np.float32).ravel()
highs  = df["High"].values.astype(np.float32).ravel()
lows   = df["Low"].values.astype(np.float32).ravel()

LOOKBACK  = 50
LOOKAHEAD = 10
N = len(closes)

[*********************100%***********************]  1 of 1 completed


In [15]:
# ── Discrete body levels ───────────────────────────────────────────────────────
body_lo = np.minimum(opens, closes)
body_hi = np.maximum(opens, closes)

# ── Step 2: Nearest past level (vectorized) ────────────────────────────────────
# For each bar i, find the past bar in [i-LOOKBACK, i) whose level
# (min distance to either lo or hi edge) is closest to closes[i]

pad_lo = np.pad(body_lo, (LOOKBACK, 0), constant_values=np.nan).ravel()
pad_hi = np.pad(body_hi, (LOOKBACK, 0), constant_values=np.nan).ravel()

win_lo = sliding_window_view(pad_lo, window_shape=LOOKBACK)[:N]  # (N, LOOKBACK)
win_hi = sliding_window_view(pad_hi, window_shape=LOOKBACK)[:N]
# ── Discrete body levels ───────────────────────────────────────────────────────
body_lo = np.minimum(opens, closes)
body_hi = np.maximum(opens, closes)


In [16]:
# ── Step 2: Nearest past level (vectorized) ────────────────────────────────────
# For each bar i, find the past bar in [i-LOOKBACK, i) whose level
# (min distance to either lo or hi edge) is closest to closes[i]

pad_lo = np.pad(body_lo, (LOOKBACK, 0), constant_values=np.nan).ravel()
pad_hi = np.pad(body_hi, (LOOKBACK, 0), constant_values=np.nan).ravel()

win_lo = sliding_window_view(pad_lo, window_shape=LOOKBACK)[:N]  # (N, LOOKBACK)
win_hi = sliding_window_view(pad_hi, window_shape=LOOKBACK)[:N]

# Distance = min(|close - lo|, |close - hi|) for each past bar
dist_lo = np.abs(closes[:, None] - win_lo)   # (N, LOOKBACK)
dist_hi = np.abs(closes[:, None] - win_hi)
dist    = np.minimum(dist_lo, dist_hi)
dist    = np.where(np.isnan(win_lo), np.inf, dist)

best = np.argmin(dist, axis=1)               # (N,)
rows = np.arange(N)

near_lo = win_lo[rows, best]
near_hi = win_hi[rows, best]
near_lo[:LOOKBACK] = np.nan
near_hi[:LOOKBACK] = np.nan

# ── Step 3: Classify events (vectorized) ──────────────────────────────────────
gain      = closes > near_hi
lose      = closes < near_lo
fail_gain = (highs > near_hi) & (closes <= near_hi) & ~lose
fail_lose = (lows  < near_lo) & (closes >= near_lo) & ~gain

event = np.full(N, None, dtype=object)
event[gain]      = "gain"
event[lose]      = "lose"
event[fail_gain] = "fail_gain"
event[fail_lose] = "fail_lose"



In [17]:
# ── Step 4: First-touch engine (vectorized) ────────────────────────────────────
# For each bar i, scan future bars to find which boundary is hit FIRST.
#
# Strategy: build (N, LOOKAHEAD) windows of future highs/lows, then for
# each bar find the minimum index where hi is breached vs lo is breached.
# Whichever index is smaller = first touch.

pad_fhi = np.pad(highs, (0, LOOKAHEAD), constant_values=np.nan).ravel()
pad_flo = np.pad(lows,  (0, LOOKAHEAD), constant_values=np.nan).ravel()

# Shift by 1 so window starts at i+1 (future, not current bar)
win_fhi = sliding_window_view(pad_fhi, window_shape=LOOKAHEAD)[1:N+1]  # (N, LOOKAHEAD)
win_flo = sliding_window_view(pad_flo, window_shape=LOOKAHEAD)[1:N+1]

# Boolean: which future bars breach each boundary
breach_up   = win_fhi > near_hi[:, None]   # (N, LOOKAHEAD)
breach_down = win_flo < near_lo[:, None]

# First index of breach (-1 if never breached) using argmax trick:
# argmax on bool array returns index of first True; if all False, returns 0
# So we mask: if no breach, set to LOOKAHEAD (sentinel = "never")
first_up   = np.where(breach_up.any(axis=1),   breach_up.argmax(axis=1),   LOOKAHEAD)
first_down = np.where(breach_down.any(axis=1), breach_down.argmax(axis=1), LOOKAHEAD)

# Compare first touch indices
outcome = np.where(
    (first_up == LOOKAHEAD) & (first_down == LOOKAHEAD), "none",
    np.where(first_up < first_down, "up_first",
    np.where(first_down < first_up, "down_first",
    "simultaneous"))   # tied on same bar
)


In [19]:
import yfinance as yf
import pandas as pd
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

# ── Load data ──────────────────────────────────────────────────────────────────
df = yf.download("NQ=F", interval="5m", period="60d", auto_adjust=True)
df.columns = df.columns.get_level_values(0)
df = df.dropna()

opens  = df["Open"].values.astype(np.float32).ravel()
closes = df["Close"].values.astype(np.float32).ravel()
highs  = df["High"].values.astype(np.float32).ravel()
lows   = df["Low"].values.astype(np.float32).ravel()

LOOKBACK  = 50
LOOKAHEAD = 10
TRAIN_RATIO = 0.7

N = len(closes)
split = int(N * TRAIN_RATIO)

print(f"Total bars:     {N}")
print(f"Train bars:     {split}  ({df.index[0]} → {df.index[split-1]})")
print(f"OOS bars:       {N - split}  ({df.index[split]} → {df.index[-1]})")

# ── Discrete body levels ───────────────────────────────────────────────────────
body_lo = np.minimum(opens, closes)
body_hi = np.maximum(opens, closes)

# ── Nearest past level (vectorized) ───────────────────────────────────────────
pad_lo = np.pad(body_lo, (LOOKBACK, 0), constant_values=np.nan).ravel()
pad_hi = np.pad(body_hi, (LOOKBACK, 0), constant_values=np.nan).ravel()

win_lo = sliding_window_view(pad_lo, window_shape=LOOKBACK)[:N]
win_hi = sliding_window_view(pad_hi, window_shape=LOOKBACK)[:N]

dist = np.minimum(
    np.abs(closes[:, None] - win_lo),
    np.abs(closes[:, None] - win_hi)
)
dist = np.where(np.isnan(win_lo), np.inf, dist)

best    = np.argmin(dist, axis=1)
rows    = np.arange(N)
near_lo = win_lo[rows, best]
near_hi = win_hi[rows, best]
near_lo[:LOOKBACK] = np.nan
near_hi[:LOOKBACK] = np.nan

# ── Classify events ────────────────────────────────────────────────────────────
gain      = closes > near_hi
lose      = closes < near_lo
fail_gain = (highs > near_hi) & (closes <= near_hi) & ~lose
fail_lose = (lows  < near_lo) & (closes >= near_lo) & ~gain

event = np.full(N, None, dtype=object)
event[gain]      = "gain"
event[lose]      = "lose"
event[fail_gain] = "fail_gain"
event[fail_lose] = "fail_lose"

# ── First-touch engine ─────────────────────────────────────────────────────────
pad_fhi = np.pad(highs, (0, LOOKAHEAD), constant_values=np.nan).ravel()
pad_flo = np.pad(lows,  (0, LOOKAHEAD), constant_values=np.nan).ravel()

win_fhi = sliding_window_view(pad_fhi, window_shape=LOOKAHEAD)[1:N+1]
win_flo = sliding_window_view(pad_flo, window_shape=LOOKAHEAD)[1:N+1]

breach_up   = win_fhi > near_hi[:, None]
breach_down = win_flo < near_lo[:, None]

first_up   = np.where(breach_up.any(axis=1),   breach_up.argmax(axis=1),   LOOKAHEAD)
first_down = np.where(breach_down.any(axis=1), breach_down.argmax(axis=1), LOOKAHEAD)

outcome = np.where(
    (first_up == LOOKAHEAD) & (first_down == LOOKAHEAD), "none",
    np.where(first_up < first_down,  "up_first",
    np.where(first_down < first_up,  "down_first",
    "simultaneous"))
)

# ── Split AFTER all computation (no leakage) ──────────────────────────────────
# Levels use only past bars, so train/OOS split on the index is clean.
# The only boundary effect: bars [split, split+LOOKBACK) reference train-period
# candle bodies for their levels — this is fine (lookback is historical).

idx = np.arange(N)

def make_mask(start, end):
    return (
        (event != None)    &
        ~np.isnan(near_lo) &
        (idx >= max(start, LOOKBACK)) &
        (idx <  min(end, N - LOOKAHEAD))
    )

train_mask = make_mask(0,     split)
oos_mask   = make_mask(split, N)

def build_df(mask):
    return pd.DataFrame({
        "event":   event[mask],
        "outcome": outcome[mask],
    })

train_df = build_df(train_mask)
oos_df   = build_df(oos_mask)

# ── Results: train only ────────────────────────────────────────────────────────
print("\n── TRAIN transition matrix ──")
train_summary = pd.crosstab(train_df["event"], train_df["outcome"], normalize="index")
print(train_summary.round(3))
print(f"Train events: {len(train_df)}")

# OOS stored but not printed — evaluate separately when ready
# print("\n── OOS transition matrix ──")
# oos_summary = pd.crosstab(oos_df["event"], oos_df["outcome"], normalize="index")
# print(oos_summary.round(3))

[*********************100%***********************]  1 of 1 completed

Total bars:     13608
Train bars:     9525  (2026-03-06 05:05:00+00:00 → 2026-04-24 16:40:00+00:00)
OOS bars:       4083  (2026-04-24 16:45:00+00:00 → 2026-05-15 19:45:00+00:00)

── TRAIN transition matrix ──
outcome    down_first   none  simultaneous  up_first
event                                               
fail_gain       0.115  0.002         0.232     0.651
fail_lose       0.370  0.001         0.442     0.187
gain            0.007  0.000         0.310     0.682
lose            0.633  0.000         0.358     0.008
Train events: 9378


In [20]:
# ── UNTESTED LEVEL TRANSITION ENGINE ──────────────────────────────────────────
# Finds nearest UNTESTED upper/lower level for each bar, then checks first touch.
# Runs on TRAIN only. Does not modify any prior code.

# ── Build full level set from train period only ────────────────────────────────
train_lo = body_lo[:split]   # (split,)
train_hi = body_hi[:split]

# ── Mark levels as "tested" at each point in time ─────────────────────────────
# A level j is "tested" at time i if any bar k < i had:
#   low[k] <= level_hi[j]  AND  high[k] >= level_lo[j]
# i.e. price overlapped the body range at some point before i.
#
# Vectorized: build (split, split) overlap matrix, then cumsum to get
# "how many times has level j been touched by bar i"

# overlap[i, j] = True if bar i's range overlaps level j's body
# shape: (split, split) — manageable at ~9500 bars (~360MB float32)
# Use bool to keep it at ~90MB

hi_2d  = train_hi[np.newaxis, :]   # (1, split) — level highs
lo_2d  = train_lo[np.newaxis, :]   # (1, split) — level lows
bar_hi = highs[:split, np.newaxis] # (split, 1) — bar highs
bar_lo = lows[:split, np.newaxis]  # (split, 1) — bar lows

overlap = (bar_hi >= lo_2d) & (bar_lo <= hi_2d)   # (split, split) bool

# cumulative touches up to (but not including) bar i
# cum_touches[i, j] = number of bars 0..i-1 that touched level j
cum_touches = np.cumsum(overlap, axis=0)           # (split, split)
# shift by 1: at bar i, we want touches BEFORE i
cum_touches = np.vstack([np.zeros((1, split), dtype=np.int32), cum_touches[:-1]])

untested = cum_touches == 0   # (split, split) — True = level j untested at bar i

# ── For each bar i, find nearest untested level above and below close ──────────
close_train = closes[:split]

# Distance matrix: close[i] vs all level midpoints
level_mid = (train_lo + train_hi) / 2.0            # (split,)
mid_2d    = level_mid[np.newaxis, :]               # (1, split)
close_2d  = close_train[:, np.newaxis]             # (split, 1)

above = level_mid[np.newaxis, :] > close_2d       # levels above current close
below = level_mid[np.newaxis, :] < close_2d       # levels below current close

dist = np.abs(close_2d - mid_2d)                  # (split, split)

# Mask out tested levels and wrong-side levels
INF = np.inf
dist_above = np.where(untested & above, dist, INF)
dist_below = np.where(untested & below, dist, INF)

# Nearest untested upper/lower level index for each bar
nearest_upper_idx = np.argmin(dist_above, axis=1)  # (split,)
nearest_lower_idx = np.argmin(dist_below, axis=1)

# Extract their lo/hi boundaries
has_upper = dist_above[np.arange(split), nearest_upper_idx] < INF
has_lower = dist_below[np.arange(split), nearest_lower_idx] < INF

upper_lo = np.where(has_upper, train_lo[nearest_upper_idx], np.nan)
upper_hi = np.where(has_upper, train_hi[nearest_upper_idx], np.nan)
lower_lo = np.where(has_lower, train_lo[nearest_lower_idx], np.nan)
lower_hi = np.where(has_lower, train_hi[nearest_lower_idx], np.nan)

# ── Classify events (same logic as before) ────────────────────────────────────
near_lo_ut = lower_lo   # current level lower boundary = nearest untested below
near_hi_ut = upper_hi   # current level upper boundary = nearest untested above

gain_ut      = closes[:split] > near_hi_ut
lose_ut      = closes[:split] < near_lo_ut
fail_gain_ut = (highs[:split] > near_hi_ut) & (closes[:split] <= near_hi_ut) & ~lose_ut
fail_lose_ut = (lows[:split]  < near_lo_ut) & (closes[:split] >= near_lo_ut) & ~gain_ut

event_ut = np.full(split, None, dtype=object)
event_ut[gain_ut]      = "gain"
event_ut[lose_ut]      = "lose"
event_ut[fail_gain_ut] = "fail_gain"
event_ut[fail_lose_ut] = "fail_lose"

# ── First-touch against untested boundaries ────────────────────────────────────
pad_fhi_ut = np.pad(highs[:split], (0, LOOKAHEAD), constant_values=np.nan).ravel()
pad_flo_ut = np.pad(lows[:split],  (0, LOOKAHEAD), constant_values=np.nan).ravel()

win_fhi_ut = sliding_window_view(pad_fhi_ut, window_shape=LOOKAHEAD)[1:split+1]
win_flo_ut = sliding_window_view(pad_flo_ut, window_shape=LOOKAHEAD)[1:split+1]

breach_up_ut   = win_fhi_ut > upper_hi[:, None]
breach_down_ut = win_flo_ut < lower_lo[:, None]

first_up_ut   = np.where(breach_up_ut.any(axis=1),   breach_up_ut.argmax(axis=1),   LOOKAHEAD)
first_down_ut = np.where(breach_down_ut.any(axis=1), breach_down_ut.argmax(axis=1), LOOKAHEAD)

outcome_ut = np.where(
    (first_up_ut == LOOKAHEAD) & (first_down_ut == LOOKAHEAD), "none",
    np.where(first_up_ut < first_down_ut,  "up_first",
    np.where(first_down_ut < first_up_ut,  "down_first",
    "simultaneous"))
)

# ── Filter valid bars & results ────────────────────────────────────────────────
idx_tr = np.arange(split)
valid_ut = (
    (event_ut != None) &
    has_upper & has_lower &
    (idx_tr >= LOOKBACK) &
    (idx_tr <  split - LOOKAHEAD)
)

res_ut = pd.DataFrame({
    "event":   event_ut[valid_ut],
    "outcome": outcome_ut[valid_ut],
})

print("── UNTESTED LEVEL TRANSITION MATRIX (train) ──")
summary_ut = pd.crosstab(res_ut["event"], res_ut["outcome"], normalize="index")
print(summary_ut.round(3))
print(f"Events: {len(res_ut)}")

── UNTESTED LEVEL TRANSITION MATRIX (train) ──
outcome    down_first   none  simultaneous  up_first
event                                               
fail_gain       0.043  0.043         0.391     0.522
fail_lose       0.457  0.058         0.370     0.116
Events: 161


In [21]:
# ── MULTI-TIMEFRAME CONFLUENCE ENGINE ─────────────────────────────────────────
# Downloads 15m, 1h, 2h, 4h data, builds body levels per TF, then for each
# 5m bar counts how many TF levels cluster within ATR-based tolerance.
# Compares first-touch transition probs by confluence score (0–4 TFs aligned).

import yfinance as yf
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

# ── Load higher TF data ────────────────────────────────────────────────────────
tfs = {"15m": "15m", "1h": "60m", "2h": "2h", "4h": "4h"}

htf_data = {}
for name, interval in tfs.items():
    d = yf.download("NQ=F", interval=interval, period="60d", auto_adjust=True)
    d.columns = d.columns.get_level_values(0)
    d = d.dropna()
    htf_data[name] = d

# ── Build body level arrays per TF ────────────────────────────────────────────
# Each TF gives us arrays of (lo, hi, mid) for every candle
htf_levels = {}
for name, d in htf_data.items():
    o = d["Open"].values.astype(np.float32).ravel()
    c = d["Close"].values.astype(np.float32).ravel()
    ts = d.index  # timestamps for alignment
    lo = np.minimum(o, c)
    hi = np.maximum(o, c)
    mid = (lo + hi) / 2.0
    htf_levels[name] = {"lo": lo, "hi": hi, "mid": mid, "ts": ts}

# ── ATR on 5m (14-period, already have highs/lows/closes from before) ─────────
def compute_atr(highs, lows, closes, period=14):
    tr = np.maximum(highs[1:] - lows[1:],
         np.maximum(np.abs(highs[1:] - closes[:-1]),
                    np.abs(lows[1:]  - closes[:-1])))
    tr = np.concatenate([[tr[0]], tr])
    atr = np.full(len(tr), np.nan)
    atr[period] = tr[:period+1].mean()
    alpha = 1.0 / period
    for i in range(period+1, len(tr)):
        atr[i] = alpha * tr[i] + (1 - alpha) * atr[i-1]
    return atr

atr_5m = compute_atr(highs, lows, closes)   # shape (N,) on 5m bars

# ── Align HTF levels to 5m timestamps ─────────────────────────────────────────
# For each 5m bar, collect all HTF level midpoints that existed BEFORE that bar
# (i.e. HTF candle closed before 5m bar opens — no lookahead)

base_ts = df.index   # 5m timestamps, length N

def get_htf_mids_at(htf_name, base_timestamps):
    """
    Returns list of arrays: for each base bar i, the array of HTF level mids
    that had closed strictly before base_timestamps[i].
    Only uses train period HTF bars.
    """
    d    = htf_levels[htf_name]
    ts   = pd.DatetimeIndex(d["ts"])
    mids = d["mid"]

    # For each base bar, how many HTF bars have closed before it?
    # Use searchsorted on HTF timestamps
    counts = np.searchsorted(ts, base_timestamps, side="left")  # (N,)
    return mids, counts   # we'll use counts as slice endpoints

htf_info = {name: get_htf_mids_at(name, base_ts) for name in tfs}

# ── Count confluence at each 5m bar (train only) ──────────────────────────────
# For each bar i, and each TF, check if ANY prior HTF level mid falls within
# ATR_MULT * ATR_5m[i] of closes[i]. If yes → that TF contributes +1 to score.

ATR_MULT  = 0.5   # tolerance = 0.5 * ATR. Tune this.
LOOKBACK_HTF = 20  # only look at last N HTF candles (avoids stale levels)

confluence = np.zeros(split, dtype=np.int32)

for name in tfs:
    mids, counts = htf_info[name]
    tol = ATR_MULT * atr_5m[:split]   # (split,)

    for i in range(split):
        n_htf = counts[i]
        if n_htf == 0:
            continue
        # last LOOKBACK_HTF HTF bars before this 5m bar
        start = max(0, n_htf - LOOKBACK_HTF)
        recent_mids = mids[start:n_htf]
        dist = np.abs(closes[i] - recent_mids)
        if (dist < tol[i]).any():
            confluence[i] += 1

# ── Reuse events + outcomes from earlier cell ─────────────────────────────────
# `event` and `outcome` arrays already computed on full N bars above.
# We just condition them on confluence score here.

idx_tr  = np.arange(split)
base_valid = (
    (event[:split] != None) &
    ~np.isnan(near_lo[:split]) &
    (idx_tr >= LOOKBACK) &
    (idx_tr <  split - LOOKAHEAD)
)

res_mtf = pd.DataFrame({
    "event":       event[:split][base_valid],
    "outcome":     outcome[:split][base_valid],
    "confluence":  confluence[base_valid],
})

# ── Transition matrix conditioned on confluence score ─────────────────────────
print("── MULTI-TF CONFLUENCE RESULTS (train) ──\n")

for score in sorted(res_mtf["confluence"].unique()):
    subset = res_mtf[res_mtf["confluence"] == score]
    if len(subset) < 30:   # skip tiny bins
        continue
    tf_names = list(tfs.keys())
    print(f"Confluence = {score}/4 TFs  (n={len(subset)})")
    ct = pd.crosstab(subset["event"], subset["outcome"], normalize="index")
    print(ct.round(3))
    print()

# ── Summary: up_first rate for gain events by confluence ──────────────────────
print("── GAIN → up_first rate by confluence score ──")
gain_rows = res_mtf[res_mtf["event"] == "gain"]
print(gain_rows.groupby("confluence")["outcome"]
      .apply(lambda x: (x == "up_first").mean())
      .rename("up_first_rate")
      .round(3))

print("\n── LOSE → down_first rate by confluence score ──")
lose_rows = res_mtf[res_mtf["event"] == "lose"]
print(lose_rows.groupby("confluence")["outcome"]
      .apply(lambda x: (x == "down_first").mean())
      .rename("down_first_rate")
      .round(3))

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed$NQ=F: possibly delisted; no price data found  (period=60d) (Yahoo error = "Invalid input - interval=2h is not supported. Valid intervals: , 2m, 5m, 15m, 30m, 60m, 90m, 1h, 4h, 1d, 5d, 1wk, 1mo, 3mo]")


1 Failed download:
['NQ=F']: possibly delisted; no price data found  (period=60d) (Yahoo error = "Invalid input - interval=2h is not supported. Valid intervals: , 2m, 5m, 15m, 30m, 60m, 90m, 1h, 4h, 1d, 5d, 1wk, 1mo, 3mo]")
[*********************100%***********************]  1 of 1 completed


── MULTI-TF CONFLUENCE RESULTS (train) ──

Confluence = 0/4 TFs  (n=648)
outcome    down_first  none  simultaneous  up_first
event                                              
fail_gain       0.120  0.02         0.140     0.720
fail_lose       0.431  0.00         0.333     0.236
gain            0.000  0.00         0.169     0.831
lose            0.811  0.00         0.189     0.000

Confluence = 1/4 TFs  (n=1966)
outcome    down_first   none  simultaneous  up_first
event                                               
fail_gain       0.133  0.004         0.205     0.658
fail_lose       0.377  0.002         0.412     0.210
gain            0.003  0.000         0.298     0.699
lose            0.686  0.000         0.308     0.005

Confluence = 2/4 TFs  (n=3596)
outcome    down_first   none  simultaneous  up_first
event                                               
fail_gain       0.121  0.002         0.238     0.639
fail_lose       0.368  0.000         0.442     0.190
gain            0.009

In [24]:
# ── OOS VALIDATION ────────────────────────────────────────────────────────────
# Tests whether the base gain/lose momentum signal holds on unseen data.
# Uses event/outcome arrays already computed — just shifts the index window.

idx_oos = np.arange(N)

oos_valid = (
    (event != None)    &
    ~np.isnan(near_lo) &
    (idx_oos >= split + LOOKBACK) &   # fresh lookback after split
    (idx_oos <  N - LOOKAHEAD)
)

res_oos = pd.DataFrame({
    "event":   event[oos_valid],
    "outcome": outcome[oos_valid],
})

print("── OOS TRANSITION MATRIX ──")
oos_summary = pd.crosstab(res_oos["event"], res_oos["outcome"], normalize="index")
print(oos_summary.round(3))
print(f"OOS events: {len(res_oos)}\n")

# ── Compare train vs OOS side by side ─────────────────────────────────────────
print("── TRAIN vs OOS: gain → up_first ──")
train_gain_rate = (train_df[train_df["event"] == "gain"]["outcome"] == "up_first").mean()
oos_gain_rate   = (res_oos[res_oos["event"]   == "gain"]["outcome"] == "up_first").mean()
print(f"  Train: {train_gain_rate:.3f}")
print(f"  OOS:   {oos_gain_rate:.3f}")
print(f"  Decay: {train_gain_rate - oos_gain_rate:+.3f}\n")

print("── TRAIN vs OOS: lose → down_first ──")
train_lose_rate = (train_df[train_df["event"] == "lose"]["outcome"] == "down_first").mean()
oos_lose_rate   = (res_oos[res_oos["event"]   == "lose"]["outcome"] == "down_first").mean()
print(f"  Train: {train_lose_rate:.3f}")
print(f"  OOS:   {oos_lose_rate:.3f}")
print(f"  Decay: {train_lose_rate - oos_lose_rate:+.3f}\n")

# ── Binomial significance test ─────────────────────────────────────────────────
# Tests whether OOS rate is statistically different from 50/50 null hypothesis.
# If p < 0.05, the edge is unlikely to be random.
from scipy.stats import binomtest

for label, subset, col in [
    ("gain → up_first",   res_oos[res_oos["event"] == "gain"], "up_first"),
    ("lose → down_first", res_oos[res_oos["event"] == "lose"], "down_first"),
]:
    n      = len(subset)
    k      = (subset["outcome"] == col).sum()
    rate   = k / n if n > 0 else 0
    result = binomtest(k, n, p=0.5, alternative="greater")
    print(f"── {label} ──")
    print(f"  OOS n={n}, rate={rate:.3f}, p={result.pvalue:.4f}",
          "✓ significant" if result.pvalue < 0.05 else "✗ not significant")

── OOS TRANSITION MATRIX ──
outcome    down_first   none  simultaneous  up_first
event                                               
fail_gain       0.116  0.000         0.220     0.664
fail_lose       0.356  0.002         0.444     0.197
gain            0.006  0.000         0.316     0.678
lose            0.632  0.000         0.352     0.015
OOS events: 3986

── TRAIN vs OOS: gain → up_first ──
  Train: 0.682
  OOS:   0.678
  Decay: +0.004

── TRAIN vs OOS: lose → down_first ──
  Train: 0.633
  OOS:   0.632
  Decay: +0.001

── gain → up_first ──
  OOS n=1134, rate=0.678, p=0.0000 ✓ significant
── lose → down_first ──
  OOS n=982, rate=0.632, p=0.0000 ✓ significant


In [23]:
%pip install scipy

   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   -------- ------------------------------- 8.1/37.3 MB 40.6 MB/s eta 0:00:01
   ----------------- ---------------------- 16.8/37.3 MB 42.0 MB/s eta 0:00:01
   ----------------------------- ---------- 27.3/37.3 MB 44.9 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.3 MB 46.3 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.3 MB 46.3 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.3 MB 46.3 MB/s eta 0:00:01
   ---------------------------------------- 37.3/37.3 MB 25.1 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


In [28]:
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

# ─────────────────────────────────────────────────────────────
LOOKAHEAD_R = 20
EPS = 1e-6
N = len(closes)

# ── FUTURE WINDOWS ───────────────────────────────────────────
pad_hi = np.pad(highs, (0, LOOKAHEAD_R), constant_values=np.nan)
pad_lo = np.pad(lows,  (0, LOOKAHEAD_R), constant_values=np.nan)

win_hi = sliding_window_view(pad_hi, LOOKAHEAD_R)[1:N+1]
win_lo = sliding_window_view(pad_lo, LOOKAHEAD_R)[1:N+1]

# ── FILTER EVENTS ─────────────────────────────────────────────
oos_idx = np.where(oos_valid)[0]
valid_mask = np.isin(np.arange(N), oos_idx) & np.isin(event, ["gain", "lose"])
idx = np.where(valid_mask)[0]

ev     = event[idx]
entry  = closes[idx]
nh     = near_hi[idx]
nl     = near_lo[idx]

width = np.maximum(nh - nl, EPS)   # FIX: prevents zero-risk explosion

fhi = win_hi[idx]
flo = win_lo[idx]

# ─────────────────────────────────────────────────────────────
# GAIN CASE
# ─────────────────────────────────────────────────────────────
gain_mask = ev == "gain"

g_fhi = fhi[gain_mask]
g_flo = flo[gain_mask]
g_entry = entry[gain_mask]
g_target = nh[gain_mask]
g_width = width[gain_mask]

g_hit = g_fhi > g_target[:, None]
g_hit_any = g_hit.any(axis=1)

g_first = np.argmax(g_hit, axis=1)

g_reward = np.full(len(g_fhi), np.nan)
g_mae = np.full(len(g_fhi), np.nan)

valid = g_hit_any

g_reward[valid] = np.maximum(
    0,
    g_fhi[valid, g_first[valid]] - g_target[valid]
)

# MAE (kept simple but stable)
for i in np.where(valid)[0]:
    g_mae[i] = np.nanmax(
        g_entry[i] - g_flo[i, :g_first[i] + 1]
    )

# ─────────────────────────────────────────────────────────────
# LOSE CASE
# ─────────────────────────────────────────────────────────────
lose_mask = ev == "lose"

l_fhi = fhi[lose_mask]
l_flo = flo[lose_mask]
l_entry = entry[lose_mask]
l_target = nl[lose_mask]
l_width = width[lose_mask]

l_hit = l_flo < l_target[:, None]
l_hit_any = l_hit.any(axis=1)

l_first = np.argmax(l_hit, axis=1)

l_reward = np.full(len(l_flo), np.nan)
l_mae = np.full(len(l_flo), np.nan)

valid = l_hit_any

l_reward[valid] = np.maximum(
    0,
    l_target[valid] - l_flo[valid, l_first[valid]]
)

for i in np.where(valid)[0]:
    l_mae[i] = np.nanmax(
        l_fhi[i, :l_first[i] + 1] - l_entry[i]
    )

# ─────────────────────────────────────────────────────────────
# BUILD DATAFRAME
# ─────────────────────────────────────────────────────────────
r_df = pd.concat([
    pd.DataFrame({
        "event": "gain",
        "reward": g_reward,
        "mae": g_mae,
        "risk_unit": g_width
    }),
    pd.DataFrame({
        "event": "lose",
        "reward": l_reward,
        "mae": l_mae,
        "risk_unit": l_width
    })
], ignore_index=True)

# ─────────────────────────────────────────────────────────────
# CLEAN NUMERICS (CRITICAL FIX)
# ─────────────────────────────────────────────────────────────
r_df = r_df.replace([np.inf, -np.inf], np.nan)
r_df = r_df[r_df["risk_unit"] > EPS]

# ─────────────────────────────────────────────────────────────
# R METRICS
# ─────────────────────────────────────────────────────────────
r_df["R"] = r_df["reward"] / r_df["risk_unit"]
r_df["mae_R"] = r_df["mae"] / r_df["risk_unit"]

# ─────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────
print("── R-MULTIPLE SUMMARY ──\n")

for ev in ["gain", "lose"]:
    sub = r_df[r_df["event"] == ev]
    hit = sub[sub["reward"].notna()]
    miss = sub[sub["reward"].isna()]

    hit_rate = len(hit) / len(sub)

    print(f"Event: {ev} (n={len(sub)})")
    print(f"  Hit rate:           {hit_rate:.3f}")
    print(f"  Avg R (hits):       {hit['R'].mean():.3f}")
    print(f"  Avg MAE-R (hits):   {hit['mae_R'].mean():.3f}")
    print(f"  Avg MAE-R (miss):   {miss['mae_R'].mean():.3f}")
    print(f"  Median R (hits):    {hit['R'].median():.3f}")

    expectancy = (hit_rate * hit["R"].mean()) - ((1 - hit_rate) * miss["mae_R"].mean())
    print(f"  Expectancy (R):     {expectancy:.3f}\n")

# ─────────────────────────────────────────────────────────────
# DISTRIBUTION
# ─────────────────────────────────────────────────────────────
hits_all = r_df[r_df["reward"].notna()]

print("── R DISTRIBUTION ──")
for p in [10, 25, 50, 75, 90]:
    print(f"p{p}: {np.percentile(hits_all['R'], p):.3f}")

print("\nMean R:", hits_all["R"].mean())
print("% > 1R:", (hits_all["R"] > 1).mean())
print("% > 2R:", (hits_all["R"] > 2).mean())
print("% > 3R:", (hits_all["R"] > 3).mean())

── R-MULTIPLE SUMMARY ──

Event: gain (n=1131)
  Hit rate:           0.997
  Avg R (hits):       3.923
  Avg MAE-R (hits):   2.647
  Avg MAE-R (miss):   nan
  Median R (hits):    1.379
  Expectancy (R):     nan

Event: lose (n=977)
  Hit rate:           0.994
  Avg R (hits):       3.890
  Avg MAE-R (hits):   2.676
  Avg MAE-R (miss):   nan
  Median R (hits):    1.280
  Expectancy (R):     nan

── R DISTRIBUTION ──
p10: 0.255
p25: 0.592
p50: 1.318
p75: 3.162
p90: 7.167

Mean R: 3.9079857172735557
% > 1R: 0.5878989995235827
% > 2R: 0.37208194378275367
% > 3R: 0.26107670319199616


# ── FIX: R-MULTIPLE WITH PROPER NEXT-LEVEL TARGETS ───────────────────────────
# For gain events: entry level = near_hi (already broken)
#                  target      = nearest body high ABOVE near_hi
# For lose events: entry level = near_lo (already broken)  
#                  target      = nearest body low BELOW near_lo

LOOKAHEAD_R = 20
EPS = 1e-6

# ── Find next level above/below for each bar ───────────────────────────────────
# Reuse win_lo/win_hi from the rolling nearest calculation (past LOOKBACK bars)
# For gain: find nearest past body HIGH that is above near_hi[i]
# For lose: find nearest past body LOW  that is below near_lo[i]

pad_lo2 = np.pad(body_lo, (LOOKBACK, 0), constant_values=np.nan).ravel()
pad_hi2 = np.pad(body_hi, (LOOKBACK, 0), constant_values=np.nan).ravel()

win_lo2 = sliding_window_view(pad_lo2, window_shape=LOOKBACK)[:N]
win_hi2 = sliding_window_view(pad_hi2, window_shape=LOOKBACK)[:N]

# Next level UP = nearest body high above near_hi
dist_up = np.where(
    win_hi2 > near_hi[:, None],
    win_hi2 - near_hi[:, None],
    np.inf
)
next_up_idx = np.argmin(dist_up, axis=1)
next_up      = win_hi2[np.arange(N), next_up_idx]
next_up      = np.where(dist_up[np.arange(N), next_up_idx] < np.inf, next_up, np.nan)

# Next level DOWN = nearest body low below near_lo
dist_dn = np.where(
    win_lo2 < near_lo[:, None],
    near_lo[:, None] - win_lo2,
    np.inf
)
next_dn_idx = np.argmin(dist_dn, axis=1)
next_dn      = win_lo2[np.arange(N), next_dn_idx]
next_dn      = np.where(dist_dn[np.arange(N), next_dn_idx] < np.inf, next_dn, np.nan)

# ── Build future windows ───────────────────────────────────────────────────────
pad_fhi2 = np.pad(highs, (0, LOOKAHEAD_R), constant_values=np.nan).ravel()
pad_flo2 = np.pad(lows,  (0, LOOKAHEAD_R), constant_values=np.nan).ravel()

win_fhi2 = sliding_window_view(pad_fhi2, window_shape=LOOKAHEAD_R)[1:N+1]
win_flo2 = sliding_window_view(pad_flo2, window_shape=LOOKAHEAD_R)[1:N+1]

# ── Filter OOS gain/lose with valid next levels ────────────────────────────────
oos_idx   = np.where(oos_valid)[0]
valid_mask = (
    np.isin(np.arange(N), oos_idx) &
    np.isin(event, ["gain", "lose"]) &
    ~np.isnan(next_up) &
    ~np.isnan(next_dn)
)
idx2 = np.where(valid_mask)[0]

ev2      = event[idx2]
entry2   = closes[idx2]
nh2      = near_hi[idx2]
nl2      = near_lo[idx2]
tgt_up   = next_up[idx2]    # real target for gain
tgt_dn   = next_dn[idx2]    # real target for lose
fhi2     = win_fhi2[idx2]
flo2     = win_flo2[idx2]

# ── Gain: risk = close - near_lo (stop below broken level) ────────────────────
gain_mask2  = ev2 == "gain"
g_entry     = entry2[gain_mask2]
g_stop      = nl2[gain_mask2]        # stop = body low of reference candle
g_target    = tgt_up[gain_mask2]     # target = next level up
g_fhi       = fhi2[gain_mask2]
g_flo       = flo2[gain_mask2]

g_risk      = np.maximum(g_entry - g_stop, EPS)
g_hit       = g_fhi > g_target[:, None]
g_hit_any   = g_hit.any(axis=1)
g_first     = np.argmax(g_hit, axis=1)

g_reward    = np.full(len(g_fhi), np.nan)
g_mae       = np.full(len(g_fhi), np.nan)

for i in np.where(g_hit_any)[0]:
    g_reward[i] = g_target[i] - g_entry[i]
    g_mae[i]    = np.nanmax(g_entry[i] - g_flo[i, :g_first[i]+1]) if g_first[i] > 0 else 0

for i in np.where(~g_hit_any)[0]:
    g_mae[i] = np.nanmax(g_entry[i] - g_flo[i])

# ── Lose: risk = near_hi - close (stop above broken level) ────────────────────
lose_mask2  = ev2 == "lose"
l_entry     = entry2[lose_mask2]
l_stop      = nh2[lose_mask2]        # stop = body high of reference candle
l_target    = tgt_dn[lose_mask2]     # target = next level down
l_fhi       = fhi2[lose_mask2]
l_flo       = flo2[lose_mask2]

l_risk      = np.maximum(l_stop - l_entry, EPS)
l_hit       = l_flo < l_target[:, None]
l_hit_any   = l_hit.any(axis=1)
l_first     = np.argmax(l_hit, axis=1)

l_reward    = np.full(len(l_flo), np.nan)
l_mae       = np.full(len(l_flo), np.nan)

for i in np.where(l_hit_any)[0]:
    l_reward[i] = l_entry[i] - l_target[i]
    l_mae[i]    = np.nanmax(l_fhi[i, :l_first[i]+1] - l_entry[i]) if l_first[i] > 0 else 0

for i in np.where(~l_hit_any)[0]:
    l_mae[i] = np.nanmax(l_fhi[i] - l_entry[i])

# ── Build dataframe ────────────────────────────────────────────────────────────
r_df2 = pd.concat([
    pd.DataFrame({"event": "gain", "reward": g_reward, "mae": g_mae, "risk": g_risk}),
    pd.DataFrame({"event": "lose", "reward": l_reward, "mae": l_mae, "risk": l_risk}),
], ignore_index=True)

r_df2 = r_df2.replace([np.inf, -np.inf], np.nan)
r_df2["R"]     = r_df2["reward"] / r_df2["risk"]
r_df2["mae_R"] = r_df2["mae"]    / r_df2["risk"]

# ── Summary ────────────────────────────────────────────────────────────────────
print("── R-MULTIPLE SUMMARY (fixed targets, OOS) ──\n")

for ev in ["gain", "lose"]:
    sub  = r_df2[r_df2["event"] == ev]
    hit  = sub[sub["reward"].notna()]
    miss = sub[sub["reward"].isna()]
    hit_rate = len(hit) / len(sub)

    avg_r_win  = hit["R"].mean()
    avg_r_loss = miss["mae_R"].mean() if len(miss) > 0 else 0

    expectancy = (hit_rate * avg_r_win) - ((1 - hit_rate) * avg_r_loss)

    print(f"Event: {ev}  (n={len(sub)})")
    print(f"  Hit rate:           {hit_rate:.3f}")
    print(f"  Avg R (hits):       {avg_r_win:.3f}")
    print(f"  Avg MAE-R (hits):   {hit['mae_R'].mean():.3f}")
    print(f"  Avg MAE-R (miss):   {avg_r_loss:.3f}")
    print(f"  Median R (hits):    {hit['R'].median():.3f}")
    print(f"  Expectancy (R):     {expectancy:.3f}",
          "✓" if expectancy > 0 else "✗")
    print()

print("── R DISTRIBUTION (hits, both events) ──")
hits_all2 = r_df2[r_df2["reward"].notna()]
for p in [10, 25, 50, 75, 90]:
    print(f"  p{p:02d}: {np.percentile(hits_all2['R'], p):.3f}R")
print(f"\n  Mean R:     {hits_all2['R'].mean():.3f}")
print(f"  % > 1R:     {(hits_all2['R'] > 1).mean():.3f}")
print(f"  % > 2R:     {(hits_all2['R'] > 2).mean():.3f}")
print(f"  % > 3R:     {(hits_all2['R'] > 3).mean():.3f}")